# 训练 Non-IID 模型

这个 notebook 用 pair-feature 回归器训练 nonrenewal / non-i.i.d. waiting-time 轨迹。

估计器使用一阶马尔可夫情形下最小的特征升级：

```text
单点 probes:       mean_n exp(-s_k tau_n)
相邻 pair probes:  mean_n exp(-s_m tau_n - u_m tau_{n+1})
```

学习到的 readout 对这些经验特征是 affine 的。这样既保留了训练后的显式线性化，也让估计器能看到相邻跃迁结构。


## 导入与全局设置

这一格定义 dtype、随机种子、设备、线程数，以及 notebook 里共用的一些常数。


In [ ]:
import json
import math
import os
import time
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

PAD_VALUE = 0.0
NP_DTYPE = np.float32
TRAIN_DTYPE = torch.float32
EPS = 1e-8
my_seed_number = int(globals().get("my_seed_number", 42))

np.random.seed(my_seed_number)
torch.manual_seed(my_seed_number)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(my_seed_number)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"
cpu_budget = max(1, int(os.environ.get("SLURM_CPUS_PER_TASK", "1")))
torch.set_num_threads(cpu_budget)
torch.set_num_interop_threads(max(1, min(4, cpu_budget)))

print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CPU threads: {cpu_budget}")


## 数据与训练配置

默认数据根目录是 `data`。如果 job 脚本把数据 staged 到 scratch，可以用 `PARAMEST_DATA_ROOT` 覆盖。

这个 notebook 接受：

- 目标为 `[delta, omega]` 的三能级 nonrenewal 模型


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {"notebooks", "train_notebooks"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_KIND = "noniid"

if "model_name" not in globals():
    model_name = os.environ.get("PARAMEST_MODEL_NAME", "v").strip().lower()
if "retrain" not in globals():
    retrain = bool(int(os.environ.get("PARAMEST_RETRAIN", "1")))
if "load_training_data" not in globals():
    load_training_data = True

base_data_root = PROJECT_ROOT / "data"
runtime_data_root = Path(os.environ.get("PARAMEST_DATA_ROOT", str(base_data_root))).resolve()
if not runtime_data_root.exists():
    raise FileNotFoundError(f"runtime data root not found: {runtime_data_root}")

dataset_root = runtime_data_root / "tragectories" / model_name
path_tau = dataset_root / "trajectories.npy"
path_param = dataset_root / "params.npy"
path_meta = dataset_root / "metadata.json"

L_max = int(globals().get("L_max", 50))
target_ntraj = int(globals().get("target_ntraj", 1_000_000))
default_batch_size = 8192 if device.type == "cuda" else 512
batch_size = int(globals().get("batch_size", default_batch_size))
num_epochs = int(globals().get("num_epochs", 80 if device.type == "cuda" else 4))
learning_rate = float(globals().get("learning_rate", 2e-3))
weight_decay = float(globals().get("weight_decay", 1e-4))
train_fraction = float(globals().get("train_fraction", 0.8))
train_length_choices = list(globals().get("train_length_choices", [10, 15, 20, 30, 40, 50]))
grad_clip_norm = float(globals().get("grad_clip_norm", 1.0))
log_every = int(globals().get("log_every", 5 if device.type == "cuda" else 1))
save_checkpoint = bool(globals().get("save_checkpoint", True))
loader_workers = max(0, min(16, cpu_budget - 1))
loader_prefetch = int(globals().get("loader_prefetch_factor", 4))

loader_kwargs = {
    "num_workers": loader_workers,
    "pin_memory": pin_memory,
}
if loader_workers > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = max(2, loader_prefetch)

checkpoint_dir = base_data_root / "models" / model_name
checkpoint_dir.mkdir(parents=True, exist_ok=True)
model_ckpt_path = checkpoint_dir / f"noniid_pair_laplace_{model_name}.pt"
ckpt_for_load = torch.load(model_ckpt_path, map_location="cpu") if model_ckpt_path.exists() else None

metadata = json.loads(path_meta.read_text()) if path_meta.exists() else {}
print("model_name =", model_name)
print("dataset_kind =", DATASET_KIND)
print("dataset_root =", dataset_root)
print("checkpoint =", model_ckpt_path)
print("metadata sampling_mode =", metadata.get("sampling_mode", metadata.get("generator", "unknown")))


### 加载并划分数据

这里按参数块划分数据：同一个参数点下的轨迹会留在同一侧，避免相同参数点同时出现在训练集和验证集里造成泄漏。


In [ ]:
def _choose_target_columns(param_array: np.ndarray):
    if "target_columns" in globals() and target_columns is not None:
        return np.asarray(target_columns, dtype=np.int64)
    env_cols = os.environ.get("PARAMEST_TARGET_COLUMNS")
    if env_cols:
        return np.asarray([int(x) for x in env_cols.split(",") if x.strip()], dtype=np.int64)
    return np.arange(param_array.shape[1], dtype=np.int64)


def paramblocks(y_raw: np.ndarray) -> np.ndarray:
    y_raw = np.asarray(y_raw)
    if len(y_raw) == 0:
        return np.zeros((0, 2), dtype=np.int64)
    changes = np.any(y_raw[1:] != y_raw[:-1], axis=1)
    starts = np.r_[0, np.flatnonzero(changes) + 1]
    stops = np.r_[starts[1:], len(y_raw)]
    return np.stack([starts, stops], axis=1).astype(np.int64, copy=False)


def splitblocks(taus_raw, y_raw, train_frac=0.8, seed=42):
    blocks = paramblocks(y_raw)
    if len(blocks) < 2:
        raise ValueError("Need at least two parameter blocks for train/val split.")

    rng = np.random.default_rng(seed)
    order = rng.permutation(len(blocks))
    n_train = int(round(train_frac * len(blocks)))
    n_train = min(max(n_train, 1), len(blocks) - 1)

    train_blocks = blocks[order[:n_train]]
    val_blocks = blocks[order[n_train:]]

    def gather(arr, chosen_blocks):
        parts = [np.asarray(arr[start:stop]) for start, stop in chosen_blocks]
        return np.concatenate(parts, axis=0)

    x_train = gather(taus_raw, train_blocks)
    y_train = gather(y_raw, train_blocks)
    x_val = gather(taus_raw, val_blocks)
    y_val = gather(y_raw, val_blocks)

    info = {
        "n_blocks": int(len(blocks)),
        "train_blocks": int(len(train_blocks)),
        "val_blocks": int(len(val_blocks)),
        "train_rows": int(len(x_train)),
        "val_rows": int(len(x_val)),
    }
    return x_train, x_val, y_train, y_val, info


def as_sequence_dataset(taus_raw, y_raw):
    taus_raw = np.asarray(taus_raw, dtype=NP_DTYPE)
    y_raw = np.asarray(y_raw, dtype=NP_DTYPE)
    valid_mask = np.isfinite(taus_raw) & (taus_raw > 0.0)
    lengths = np.sum(valid_mask, axis=1).astype(np.int64)
    if np.any(lengths == 0):
        raise ValueError("Found trajectories with zero valid waiting times after preprocessing.")
    taus = np.where(valid_mask, taus_raw, PAD_VALUE).astype(NP_DTYPE, copy=False)
    return {
        "taus": torch.from_numpy(taus),
        "mask": torch.from_numpy(valid_mask.astype(np.bool_, copy=False)),
        "lengths": torch.from_numpy(lengths),
        "y": torch.from_numpy(y_raw.astype(NP_DTYPE, copy=False)),
    }


def valid_log_stats(data):
    taus = data["taus"]
    mask = data["mask"]
    vals = torch.log1p(taus[mask].to(dtype=TRAIN_DTYPE))
    mean = vals.mean()
    std = vals.std(unbiased=False).clamp_min(1e-4)
    return mean, std

train_loader = None
val_loader = None
train_data = None
val_data = None
target_mean = None
target_std = None
target_dim = None
tau_log_mean = None
tau_log_std = None
split_info = None

if load_training_data:
    if not path_tau.exists():
        raise FileNotFoundError(f"trajectory file not found: {path_tau}")
    if not path_param.exists():
        raise FileNotFoundError(f"param file not found: {path_param}")

    tau_all = np.load(path_tau, mmap_mode="r")
    param_all = np.load(path_param, mmap_mode="r")
    n_total = min(tau_all.shape[0], param_all.shape[0])
    ntraj_select = min(target_ntraj, n_total)

    if ntraj_select < n_total:
        rng = np.random.default_rng(my_seed_number)
        select_idx = np.sort(rng.choice(n_total, size=ntraj_select, replace=False))
    else:
        select_idx = np.arange(n_total)

    tau_list = np.asarray(tau_all[select_idx, :L_max], dtype=NP_DTYPE)
    param_list_all = np.asarray(param_all[select_idx], dtype=NP_DTYPE)
    target_columns_arr = _choose_target_columns(param_list_all)
    param_list = param_list_all[:, target_columns_arr]

    x_train, x_val, y_train, y_val, split_info = splitblocks(
        tau_list,
        param_list,
        train_frac=train_fraction,
        seed=my_seed_number,
    )

    train_data = as_sequence_dataset(x_train, y_train)
    val_data = as_sequence_dataset(x_val, y_val)

    target_mean = train_data["y"].to(dtype=TRAIN_DTYPE).mean(dim=0)
    target_std = train_data["y"].to(dtype=TRAIN_DTYPE).std(dim=0, unbiased=False).clamp_min(1e-6)
    target_dim = int(target_mean.numel())
    tau_log_mean, tau_log_std = valid_log_stats(train_data)

    train_dataset = TensorDataset(train_data["taus"], train_data["mask"], train_data["y"])
    val_dataset = TensorDataset(val_data["taus"], val_data["mask"], val_data["y"])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=False, **loader_kwargs)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=False, **loader_kwargs)

    print("tau_all shape =", tuple(tau_all.shape))
    print("param_all shape =", tuple(param_all.shape))
    print("target columns =", target_columns_arr.tolist())
    print("target_dim =", target_dim)
    print("selected trajectories =", ntraj_select)
    print("split_info =", split_info)
    print("target_mean =", target_mean.numpy())
    print("target_std  =", target_std.numpy())
    print("tau_log_mean/std =", float(tau_log_mean), float(tau_log_std))
elif ckpt_for_load is not None:
    target_mean = ckpt_for_load["target_mean"].to(dtype=TRAIN_DTYPE)
    target_std = ckpt_for_load["target_std"].to(dtype=TRAIN_DTYPE)
    target_dim = int(target_mean.numel())
    tau_log_mean = ckpt_for_load["tau_log_mean"].to(dtype=TRAIN_DTYPE)
    tau_log_std = ckpt_for_load["tau_log_std"].to(dtype=TRAIN_DTYPE)
else:
    raise RuntimeError("Set load_training_data=True or provide an existing checkpoint.")


## 模型定义

模型是显式经验 Laplace 统计量上的 affine readout：

1. 单点 probes 处理边缘 waiting-time law 和有限样本边界效应
2. 相邻 pair probes 处理一阶马尔可夫转移信息
3. 两个长度特征让线性 head 能看到有限样本 scaling

pair 分支使用学习到的 probe pairs `(s_m, u_m)`，而不是完整的 `K x K` 网格，因此特征维度只随 pair probe 数量线性增长。


In [ ]:
def _init_probe_values(init, K: int, dtype: torch.dtype) -> torch.Tensor:
    init_t = torch.as_tensor(init, dtype=dtype)
    if init_t.ndim == 1 and init_t.numel() == K:
        return init_t.clone()
    if init_t.ndim == 1 and init_t.numel() == 2:
        return torch.linspace(init_t[0], init_t[1], K, dtype=dtype)
    raise ValueError(f"Expected probe initializer with shape ({K},) or (2,), got {tuple(init_t.shape)}")


def _positive_raw_from_init(x: torch.Tensor, eps: float) -> torch.Tensor:
    x = torch.clamp(x - eps, min=1e-6)
    return torch.log(torch.expm1(x))


class OneAndPairLaplaceFeatures(nn.Module):
    """
    用于有序一阶轨迹的经验复 Laplace probes。

    单点分支：
        F_k = mean_n exp(-(alpha_k - i beta_k) tau_n)

    pair 分支：
        G_m = mean_n exp(-(a_m - i b_m) tau_n - (c_m - i d_m) tau_{n+1})

    输出：
        [Re(F), Im(F), Re(G), Im(G), 1/sqrt(N), 1/sqrt(N_pair)]
    """

    def __init__(
        self,
        K_one: int = 4,
        M_pair: int = 4,
        alpha_init=(0.2, 0.8),
        beta_init=(-2.0, 2.0),
        pair_alpha_init=(0.2, 0.8),
        pair_beta_init=(-2.0, 2.0),
        eps_alpha: float = 1e-4,
    ):
        super().__init__()
        self.K_one = int(K_one)
        self.M_pair = int(M_pair)
        self.eps_alpha = float(eps_alpha)

        alpha0 = _init_probe_values(alpha_init, self.K_one, TRAIN_DTYPE)
        beta0 = _init_probe_values(beta_init, self.K_one, TRAIN_DTYPE)
        self.raw_alpha = nn.Parameter(_positive_raw_from_init(alpha0, self.eps_alpha))
        self.beta = nn.Parameter(beta0)

        pair_alpha0 = _init_probe_values(pair_alpha_init, self.M_pair, TRAIN_DTYPE)
        pair_beta0 = _init_probe_values(pair_beta_init, self.M_pair, TRAIN_DTYPE)
        self.raw_pair_alpha_left = nn.Parameter(_positive_raw_from_init(pair_alpha0, self.eps_alpha))
        self.raw_pair_alpha_right = nn.Parameter(_positive_raw_from_init(torch.flip(pair_alpha0, dims=(0,)), self.eps_alpha))
        self.pair_beta_left = nn.Parameter(pair_beta0.clone())
        self.pair_beta_right = nn.Parameter(torch.flip(pair_beta0, dims=(0,)).clone())

        self.output_dim = 2 * self.K_one + 2 * self.M_pair + 2

    def one_parameters(self):
        alpha = F.softplus(self.raw_alpha) + self.eps_alpha
        return alpha, self.beta

    def pair_parameters(self):
        alpha_left = F.softplus(self.raw_pair_alpha_left) + self.eps_alpha
        alpha_right = F.softplus(self.raw_pair_alpha_right) + self.eps_alpha
        return alpha_left, self.pair_beta_left, alpha_right, self.pair_beta_right

    def forward(self, taus: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        taus = taus.to(dtype=self.raw_alpha.dtype)
        mask = mask.bool()
        mask_f = mask.to(dtype=taus.dtype)

        N = mask_f.sum(dim=1, keepdim=True).clamp_min(1.0)
        alpha, beta = self.one_parameters()
        tau = taus.unsqueeze(-1)
        one_mask = mask_f.unsqueeze(-1)
        one_amp = torch.exp(-tau * alpha)
        one_phase = tau * beta
        one_re = (one_amp * torch.cos(one_phase) * one_mask).sum(dim=1) / N
        one_im = (one_amp * torch.sin(one_phase) * one_mask).sum(dim=1) / N

        if taus.shape[1] > 1:
            left_tau = taus[:, :-1].unsqueeze(-1)
            right_tau = taus[:, 1:].unsqueeze(-1)
            pair_mask = (mask[:, :-1] & mask[:, 1:]).to(dtype=taus.dtype).unsqueeze(-1)
            N_pair = pair_mask.sum(dim=1).clamp_min(1.0)
            alpha_l, beta_l, alpha_r, beta_r = self.pair_parameters()
            pair_amp = torch.exp(-left_tau * alpha_l - right_tau * alpha_r)
            pair_phase = left_tau * beta_l + right_tau * beta_r
            pair_re = (pair_amp * torch.cos(pair_phase) * pair_mask).sum(dim=1) / N_pair
            pair_im = (pair_amp * torch.sin(pair_phase) * pair_mask).sum(dim=1) / N_pair
            pair_len = torch.rsqrt(N_pair)
        else:
            B = taus.shape[0]
            pair_re = torch.zeros((B, self.M_pair), dtype=taus.dtype, device=taus.device)
            pair_im = torch.zeros((B, self.M_pair), dtype=taus.dtype, device=taus.device)
            pair_len = torch.ones((B, 1), dtype=taus.dtype, device=taus.device)

        one_len = torch.rsqrt(N)
        return torch.cat([one_re, one_im, pair_re, pair_im, one_len, pair_len], dim=1)


class NonIIDPairLaplaceEstimator(nn.Module):
    """单点 + 相邻 pair Laplace 特征，然后接 affine readout。"""

    def __init__(
        self,
        target_dim: int,
        target_mean: torch.Tensor,
        target_std: torch.Tensor,
        K_one: int = 4,
        M_pair: int = 4,
        alpha_init=(0.2, 0.8),
        beta_init=(-2.0, 2.0),
        pair_alpha_init=(0.2, 0.8),
        pair_beta_init=(-2.0, 2.0),
    ):
        super().__init__()
        self.target_dim = int(target_dim)
        self.feat = OneAndPairLaplaceFeatures(
            K_one=K_one,
            M_pair=M_pair,
            alpha_init=alpha_init,
            beta_init=beta_init,
            pair_alpha_init=pair_alpha_init,
            pair_beta_init=pair_beta_init,
        )
        self.in_dim = self.feat.output_dim
        self.head = nn.Linear(self.in_dim, self.target_dim)

        self.register_buffer("target_mean", torch.as_tensor(target_mean, dtype=TRAIN_DTYPE))
        self.register_buffer("target_std", torch.clamp(torch.as_tensor(target_std, dtype=TRAIN_DTYPE), min=1e-6))

    @property
    def K_one(self) -> int:
        return self.feat.K_one

    @property
    def M_pair(self) -> int:
        return self.feat.M_pair

    def encode_targets(self, y: torch.Tensor) -> torch.Tensor:
        y = y.to(device=self.target_mean.device, dtype=self.target_mean.dtype)
        return (y - self.target_mean) / self.target_std

    def decode_targets(self, y_std: torch.Tensor) -> torch.Tensor:
        y_std = y_std.to(device=self.target_mean.device, dtype=self.target_mean.dtype)
        return y_std * self.target_std + self.target_mean

    def features(self, taus: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        return self.feat(taus, mask)

    def forward_standardized(self, taus: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(taus, mask))

    def forward(self, taus: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        return self.decode_targets(self.forward_standardized(taus, mask))

    def linearparams(self):
        return self.head.weight, self.head.bias

    @torch.no_grad()
    def probe_parameters(self, detach: bool = True) -> Dict[str, torch.Tensor]:
        one_alpha, one_beta = self.feat.one_parameters()
        pair_alpha_l, pair_beta_l, pair_alpha_r, pair_beta_r = self.feat.pair_parameters()
        out = {
            "one_alpha": one_alpha,
            "one_beta": one_beta,
            "pair_alpha_left": pair_alpha_l,
            "pair_beta_left": pair_beta_l,
            "pair_alpha_right": pair_alpha_r,
            "pair_beta_right": pair_beta_r,
        }
        if detach:
            out = {k: v.detach() for k, v in out.items()}
        return out


## 训练工具函数

随机截断让训练目标覆盖不同轨迹长度，和 iid notebook 里的做法一致。


In [ ]:
def mse_loss(model: NonIIDPairLaplaceEstimator, pred_std: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return F.mse_loss(pred_std, model.encode_targets(target.to(dtype=TRAIN_DTYPE)))


def truncbatch(taus: torch.Tensor, mask: torch.Tensor, length: int) -> Tuple[torch.Tensor, torch.Tensor]:
    L = int(length)
    return taus[:, :L], mask[:, :L]


def randtrunc(taus: torch.Tensor, mask: torch.Tensor, length_choices) -> Tuple[torch.Tensor, torch.Tensor, int]:
    valid_choices = [int(L) for L in length_choices if 0 < int(L) <= taus.shape[1]]
    if not valid_choices:
        raise ValueError("length_choices must contain at least one length within the current trajectory width.")
    idx = torch.randint(len(valid_choices), (1,), device=taus.device).item()
    chosen_length = valid_choices[idx]
    trunc_taus, trunc_mask = truncbatch(taus, mask, chosen_length)
    return trunc_taus, trunc_mask, chosen_length


def evalmodel(model, loader, device, length=None):
    model.eval()
    total_loss = 0.0
    total_count = 0
    total_sqerr = torch.zeros(model.target_dim, dtype=TRAIN_DTYPE, device=device)
    non_blocking = device.type == "cuda"

    with torch.no_grad():
        for taus, mask, y in loader:
            taus = taus.to(device, non_blocking=non_blocking)
            mask = mask.to(device, non_blocking=non_blocking)
            y = y.to(device, dtype=TRAIN_DTYPE, non_blocking=non_blocking)
            if length is not None:
                taus, mask = truncbatch(taus, mask, int(length))

            pred_std = model.forward_standardized(taus, mask)
            pred = model.decode_targets(pred_std)
            loss = mse_loss(model, pred_std, y)
            batch_n = y.shape[0]
            total_loss += float(loss.detach().cpu()) * batch_n
            total_count += batch_n
            total_sqerr += torch.sum((pred - y) ** 2, dim=0)

    rmse = torch.sqrt(total_sqerr / max(total_count, 1))
    return {
        "std_mse": total_loss / max(total_count, 1),
        "rmse": rmse,
    }


def evallengths(model, loader, device, lengths):
    by_length = {}
    scores = []
    for L in [int(x) for x in lengths]:
        metrics = evalmodel(model, loader, device, length=L)
        by_length[L] = metrics
        scores.append(metrics["std_mse"])
    return {
        "avg_std_mse": float(np.mean(scores)),
        "by_length": by_length,
    }


def train_model(model, train_loader, val_loader, optimizer, device, epochs, train_length_choices):
    history = {"train_std_mse": [], "val_std_mse": [], "val_rmse": [], "val_by_length": [], "epoch_timing": []}
    best_val = float("inf")
    best_state = None
    best_val_rmse = None
    best_val_by_length = None
    non_blocking = device.type == "cuda"

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        running_count = 0
        running_sqerr = torch.zeros(model.target_dim, dtype=TRAIN_DTYPE, device=device)
        chosen_lengths = []
        tick = time.perf_counter()

        for taus, mask, y in train_loader:
            taus = taus.to(device, non_blocking=non_blocking)
            mask = mask.to(device, non_blocking=non_blocking)
            y = y.to(device, dtype=TRAIN_DTYPE, non_blocking=non_blocking)
            taus, mask, chosen_length = randtrunc(taus, mask, train_length_choices)
            chosen_lengths.append(chosen_length)

            optimizer.zero_grad(set_to_none=True)
            pred_std = model.forward_standardized(taus, mask)
            pred = model.decode_targets(pred_std)
            loss = mse_loss(model, pred_std, y)
            loss.backward()
            if grad_clip_norm > 0:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
            optimizer.step()

            batch_n = y.shape[0]
            running_loss += float(loss.detach().cpu()) * batch_n
            running_count += batch_n
            running_sqerr += torch.sum((pred.detach() - y) ** 2, dim=0)

        train_std_mse = running_loss / max(running_count, 1)
        train_rmse = torch.sqrt(running_sqerr / max(running_count, 1))
        val_metrics = evallengths(model, val_loader, device, train_length_choices)
        val_std_mse = val_metrics["avg_std_mse"]
        val_by_length = {
            int(L): val_metrics["by_length"][int(L)]["rmse"].detach().cpu().tolist()
            for L in train_length_choices
        }
        val_rmse = val_metrics["by_length"][max(train_length_choices)]["rmse"].detach().cpu()
        elapsed = time.perf_counter() - tick

        history["train_std_mse"].append(train_std_mse)
        history["val_std_mse"].append(val_std_mse)
        history["val_rmse"].append(val_rmse.tolist())
        history["val_by_length"].append(val_by_length)
        history["epoch_timing"].append(elapsed)

        if val_std_mse < best_val:
            best_val = val_std_mse
            best_val_rmse = val_rmse.clone()
            best_val_by_length = val_by_length
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % log_every == 0 or epoch == 0 or epoch + 1 == epochs:
            length_counts = {int(L): chosen_lengths.count(int(L)) for L in sorted(set(chosen_lengths))}
            print(
                f"epoch {epoch + 1:03d}/{epochs} "
                f"train_std_mse={train_std_mse:.6f} "
                f"val_std_mse={val_std_mse:.6f} "
                f"train_rmse={train_rmse.detach().cpu().numpy()} "
                f"val_rmse_Lmax={val_rmse.numpy()} "
                f"lengths={length_counts} "
                f"time={elapsed:.1f}s"
            )

    if best_state is not None:
        model.load_state_dict(best_state)
    history["best_val_std_mse"] = best_val
    history["best_val_rmse"] = None if best_val_rmse is None else best_val_rmse.tolist()
    history["best_val_by_length"] = best_val_by_length
    return history


## 训练与加载

如果已有 checkpoint，并且希望直接加载它，可以设置 `retrain=False`。


In [ ]:
K_one_probes = int(globals().get("K_one_probes", globals().get("K_probes", 4)))
M_pair_probes = int(globals().get("M_pair_probes", K_one_probes))
alpha_init_range = tuple(globals().get("alpha_init_range", (0.2, 0.8)))
beta_init_range = tuple(globals().get("beta_init_range", (-2.0, 2.0)))
pair_alpha_init_range = tuple(globals().get("pair_alpha_init_range", alpha_init_range))
pair_beta_init_range = tuple(globals().get("pair_beta_init_range", beta_init_range))

if (not retrain) and (ckpt_for_load is not None):
    K_one_probes = int(ckpt_for_load.get("K_one_probes", K_one_probes))
    M_pair_probes = int(ckpt_for_load.get("M_pair_probes", M_pair_probes))

model = NonIIDPairLaplaceEstimator(
    target_dim=target_dim,
    target_mean=target_mean,
    target_std=target_std,
    K_one=K_one_probes,
    M_pair=M_pair_probes,
    alpha_init=alpha_init_range,
    beta_init=beta_init_range,
    pair_alpha_init=pair_alpha_init_range,
    pair_beta_init=pair_beta_init_range,
).to(device)

print("model =", model.__class__.__name__)
print("target_dim =", target_dim)
print("K_one_probes =", K_one_probes)
print("M_pair_probes =", M_pair_probes)
print("feature_dim =", model.in_dim)
print("batch_size =", batch_size)
print("num_epochs =", num_epochs)
print("retrain =", retrain)
print("checkpoint =", model_ckpt_path)

history = None
did_train = False
if retrain or not model_ckpt_path.exists():
    if train_loader is None or val_loader is None:
        raise RuntimeError("Training requires load_training_data=True.")
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    history = train_model(
        model,
        train_loader,
        val_loader,
        optimizer,
        device=device,
        epochs=num_epochs,
        train_length_choices=train_length_choices,
    )
    did_train = True
else:
    ckpt = ckpt_for_load if ckpt_for_load is not None else torch.load(model_ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt["model_state"])
    history = ckpt.get("history", None)
    print(f"Loaded model checkpoint from {model_ckpt_path}")

final_val = None
if val_loader is not None:
    final_val = evallengths(model, val_loader, device, train_length_choices)
    print(f"Validation standardized MSE averaged over lengths: {final_val['avg_std_mse']:.6f}")
    for L in sorted(final_val["by_length"]):
        rmse = final_val["by_length"][L]["rmse"].detach().cpu().numpy()
        print(f"Validation RMSE L={L}: {rmse}")
if history is not None and history.get("best_val_rmse") is not None:
    print("Best validation RMSE:", np.asarray(history["best_val_rmse"]))

probe_params = model.probe_parameters()
W_eff, c_eff = model.linearparams()
print("one_alpha:", probe_params["one_alpha"].detach().cpu().numpy())
print("one_beta :", probe_params["one_beta"].detach().cpu().numpy())
print("pair_alpha_left :", probe_params["pair_alpha_left"].detach().cpu().numpy())
print("pair_beta_left  :", probe_params["pair_beta_left"].detach().cpu().numpy())
print("pair_alpha_right:", probe_params["pair_alpha_right"].detach().cpu().numpy())
print("pair_beta_right :", probe_params["pair_beta_right"].detach().cpu().numpy())
print("Affine standardized weight shape:", tuple(W_eff.shape))
print("Affine standardized bias:", c_eff.detach().cpu())


## 保存当前模型

checkpoint 会包含归一化常数、模型超参数、目标列和训练历史。


In [ ]:
if did_train and save_checkpoint:
    checkpoint = {
        "model_name": model_name,
        "dataset_kind": DATASET_KIND,
        "model_class": model.__class__.__name__,
        "model_state": model.state_dict(),
        "target_mean": model.target_mean.detach().cpu(),
        "target_std": model.target_std.detach().cpu(),
        "tau_log_mean": tau_log_mean.detach().cpu(),
        "tau_log_std": tau_log_std.detach().cpu(),
        "target_dim": target_dim,
        "K_one_probes": K_one_probes,
        "M_pair_probes": M_pair_probes,
        "alpha_init_range": alpha_init_range,
        "beta_init_range": beta_init_range,
        "pair_alpha_init_range": pair_alpha_init_range,
        "pair_beta_init_range": pair_beta_init_range,
        "feature_dim": model.in_dim,
        "L_max": L_max,
        "train_length_choices": train_length_choices,
        "target_columns": target_columns_arr.tolist() if "target_columns_arr" in globals() else None,
        "history": history,
        "split_info": split_info,
        "metadata": metadata,
    }
    torch.save(checkpoint, model_ckpt_path)
    print(f"Saved checkpoint to {model_ckpt_path}")
elif not did_train:
    print("No new training run; checkpoint was not overwritten.")
else:
    print("save_checkpoint=False; checkpoint was not written.")


## 快速诊断

这里提供一个小工具，用来查看少量验证样本上的预测结果。


In [ ]:
def preview_predictions(model, loader, n: int = 5, length: Optional[int] = None):
    model.eval()
    taus, mask, y = next(iter(loader))
    taus = taus[:n].to(device)
    mask = mask[:n].to(device)
    y = y[:n].to(device, dtype=TRAIN_DTYPE)
    if length is not None:
        taus, mask = truncbatch(taus, mask, int(length))
    with torch.no_grad():
        pred = model(taus, mask)
    return {
        "target": y.detach().cpu().numpy(),
        "pred": pred.detach().cpu().numpy(),
        "error": (pred - y).detach().cpu().numpy(),
    }

if val_loader is not None:
    preview = preview_predictions(model, val_loader, n=5, length=max(train_length_choices))
    print("target:\n", preview["target"])
    print("pred:\n", preview["pred"])
    print("error:\n", preview["error"])
